# RUN_BATCH — Orchestrator batch analisa PV per tanggal

Untuk tiap tanggal, orchestrator **menjalankan seluruh template** (mis.
`20260209stringmap_v1.5.ipynb`) lalu **menyimpan notebook hasil yang sudah
ter-Run All** ke `Cek PV String/runs/YYYYMMDD.ipynb` (lengkap heatmap + output)
— **tanpa mengubah file template**. Hasil data tetap di `Cek PV String/outputs/`.

**Dua mode pengisian tanggal (boleh dicampur):**
- **Manual** — isi `DATE_TO_URL` sendiri.
- **Auto-enumerate** — beri link **folder bulanan**; subfolder hari
  (`ddmmyyyy` mis. `30012026`, atau `ddmmyy` mis. `151225`) dibaca otomatis via
  Google Drive API lalu dipetakan ke URL.

**Cara pakai:** buka di Colab -> edit sel **Config** -> **Runtime > Run all**.
Auto-enumerate butuh **1x klik izin Google** di awal.

> gdown butuh folder hari ter-share "anyone with link".

In [ ]:
# ============================ CONFIG (EDIT DI SINI) ============================
REPO_DIR     = '/content/drive/MyDrive/Cek PV String'
TEMPLATE_NB  = '/content/drive/MyDrive/Cek PV String/notebook/20260209stringmap_v1.5.ipynb'  # GANTI

RUNS_SUBDIR  = 'runs'  # subfolder di REPO_DIR utk simpan notebook hasil per tanggal
EMBED_PLOTS  = True    # True = sematkan heatmap ke notebook hasil (file lebih besar)
SKIP_IF_DONE = True    # lewati tanggal yang runs/YYYYMMDD.ipynb-nya sudah ada

# --- Mode MANUAL (boleh kosong kalau pakai auto-enumerate) -------------------
DATE_TO_URL = {
    # '2026-02-09': 'https://drive.google.com/drive/folders/17JX8...',
}

# --- Mode AUTO-ENUMERATE: cukup beri link folder BULANAN (boleh >1) ----------
MONTH_FOLDER_URLS = [
    # 'https://drive.google.com/drive/folders/<id-folder-Januari-2026>',
    # 'https://drive.google.com/drive/folders/<id-folder-Februari-2026>',
]
DATE_FROM = None   # 'YYYY-MM-DD' atau None  (batas bawah range, opsional)
DATE_TO   = None   # 'YYYY-MM-DD' atau None  (batas atas range, opsional)
# =============================================================================

In [ ]:
# =================== AUTO-ENUMERATE (jalan jika MONTH_FOLDER_URLS terisi) =====
import re as _re
from datetime import datetime as _dt, date as _date

def _folder_id(url):
    m = _re.search(r'/folders/([A-Za-z0-9_-]+)', url) or _re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    if not m:
        raise ValueError(f'Tidak bisa ambil folder id dari URL: {url}')
    return m.group(1)

def _parse_day(name):
    s = name.strip()
    if not s.isdigit():
        return None
    fmt = {8: '%d%m%Y', 6: '%d%m%y'}.get(len(s))   # 8=ddmmyyyy, 6=ddmmyy
    if not fmt:
        return None
    try:
        return _dt.strptime(s, fmt).date()
    except ValueError:
        return None

if MONTH_FOLDER_URLS:
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    _svc = build('drive', 'v3')

    _lo = _dt.strptime(DATE_FROM, '%Y-%m-%d').date() if DATE_FROM else _date.min
    _hi = _dt.strptime(DATE_TO,   '%Y-%m-%d').date() if DATE_TO   else _date.max

    _found, _skipped = {}, []
    for _murl in MONTH_FOLDER_URLS:
        _fid = _folder_id(_murl)
        _tok = None
        while True:
            _resp = _svc.files().list(
                q="'" + _fid + "' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false",
                fields='nextPageToken, files(id,name)', pageSize=1000, pageToken=_tok,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
            for _f in _resp.get('files', []):
                _d = _parse_day(_f['name'])
                if _d is None:
                    _skipped.append(_f['name']); continue
                if _lo <= _d <= _hi:
                    _found[_d.isoformat()] = 'https://drive.google.com/drive/folders/' + _f['id']
            _tok = _resp.get('nextPageToken')
            if not _tok:
                break

    # manual menang kalau bentrok; hasil diurutkan tanggal
    DATE_TO_URL = dict(sorted({**_found, **DATE_TO_URL}.items()))
    print(f'[enumerate] folder hari ketemu={len(_found)}  dilewati(nama bukan tanggal)={len(_skipped)}')
    if _skipped:
        print('  contoh dilewati:', _skipped[:5])
    print(f'[enumerate] total tanggal siap jalan = {len(DATE_TO_URL)}')
    for _k in list(DATE_TO_URL)[:3]:
        print('   ', _k, '->', DATE_TO_URL[_k])
else:
    print('[enumerate] MONTH_FOLDER_URLS kosong -> pakai DATE_TO_URL manual ('
          + str(len(DATE_TO_URL)) + ' tanggal)')

In [ ]:
# ===================== ENGINE: Run All per tanggal -> simpan .ipynb ===========
from google.colab import drive
drive.mount('/content/drive')

import os, sys, gc, io, copy, base64, contextlib, traceback
import matplotlib
import matplotlib.pyplot as plt
plt.switch_backend('Agg')        # render off-screen; gambar tetap ditangkap ke notebook
import nbformat
from nbformat.v4 import new_output

assert os.path.isdir(REPO_DIR),     f'REPO_DIR tidak ditemukan: {REPO_DIR}'
assert os.path.isfile(TEMPLATE_NB), f'TEMPLATE_NB tidak ditemukan: {TEMPLATE_NB}'
assert DATE_TO_URL, 'DATE_TO_URL kosong: isi manual atau set MONTH_FOLDER_URLS lalu run sel enumerate.'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Nonaktifkan unduhan browser Cell 8 template (TANPA mengubah file template)
import google.colab.files as _gf
_gf.download = lambda *a, **k: print('   [skip files.download]', *a)

RUNS_DIR = os.path.join(REPO_DIR, RUNS_SUBDIR)
os.makedirs(RUNS_DIR, exist_ok=True)
TEMPLATE = nbformat.read(TEMPLATE_NB, as_version=4)

_imgs = []
def _grab_figs():
    if EMBED_PLOTS:
        for _n in plt.get_fignums():
            _b = io.BytesIO()
            plt.figure(_n).savefig(_b, format='png', bbox_inches='tight')
            _imgs.append(base64.b64encode(_b.getvalue()).decode('ascii'))
    plt.close('all')

def _show_capture(*a, **k):   # tangkap tiap plt.show() (urut), lalu tutup
    _grab_figs()
plt.show = _show_capture

def _inject_url(src, url):
    out = []
    for line in src.split('\n'):
        if line.lstrip().startswith('DRIVE_FOLDER_URL') and '=' in line:
            pad = line[:len(line) - len(line.lstrip())]
            out.append(f'{pad}DRIVE_FOLDER_URL = "{url}"')
        else:
            out.append(line)
    return '\n'.join(out)

def _run_to_nb(url):
    nb = copy.deepcopy(TEMPLATE)
    g = {'__name__': '__main__'}
    n = 0
    error = None
    for cell in nb.cells:
        if cell.cell_type != 'code':
            continue
        n += 1
        cell.source = _inject_url(cell.source, url)   # rekam URL yg dipakai
        cell.execution_count = n
        _imgs.clear()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            try:
                exec(cell.source, g)
            except Exception:
                error = traceback.format_exc()
        _grab_figs()                                   # gambar yg belum di-show()
        outs = []
        if buf.getvalue():
            outs.append(new_output('stream', name='stdout', text=buf.getvalue()))
        for _b64 in _imgs:
            outs.append(new_output('display_data', data={'image/png': _b64}, metadata={}))
        if error:
            outs.append(new_output('error', ename='Error',
                                   evalue=error.strip().splitlines()[-1],
                                   traceback=error.splitlines()))
            cell.outputs = outs
            break
        cell.outputs = outs
    return nb, error

ok = skip = fail = 0
for datestr, url in DATE_TO_URL.items():
    ymd = datestr.replace('-', '')
    dest = os.path.join(RUNS_DIR, f'{ymd}.ipynb')
    if SKIP_IF_DONE and os.path.isfile(dest):
        print(f'SKIP {datestr}: {dest} sudah ada'); skip += 1; continue
    print(f'==================  {datestr}  ==================')
    nb_done, error = _run_to_nb(url)
    nbformat.write(nb_done, dest)        # selalu simpan (partial bila error)
    if error:
        print(f'FAIL {datestr}: tersimpan partial -> {dest}'); fail += 1
    else:
        print(f'OK   {datestr} -> {dest}'); ok += 1
    plt.close('all'); gc.collect()

print(f'\n=== RINGKASAN ===  sukses={ok}  skip={skip}  gagal={fail}  total={len(DATE_TO_URL)}')
print(f'Notebook per tanggal : {RUNS_DIR}/YYYYMMDD.ipynb')
print(f'Hasil data           : {os.path.join(REPO_DIR, "outputs")}/m2_findings_YYYYMMDD.xlsx (+ csv, baseline/)')